In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np
import os
from dataclasses import dataclass
import datasets
from modelscope.msdatasets import MsDataset

/opt/anaconda3/envs/rlhf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
device = "mps"
model_path = "/Users/zhangyf/llm/Qwen3-0.6B-Base"

In [2]:
# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype="auto",
    device_map="auto"
)
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)
print(model.generation_config)

NameError: name 'AutoModelForCausalLM' is not defined

In [ ]:
model.generation_config.do_sample = True
model.generation_config.eos_token_id = [151645, 151643]
model.generation_config.pad_token_id = 151643
model.generation_config.temperature = 0.7
model.generation_config.top_p = 0.8
model.generation_config.top_k = 20
model.generation_config.repetition_penalty = 1.05

print(model.generation_config)

In [ ]:
@dataclass
class SFTConfig:
    max_length: int = 2500
    batch_size: int = 2
    gradient_accumulation_steps: int = 8
    log_iter: int = 100
    max_lr: float = 2e-5
    min_lr: float = 2e-6
    warmup_steps: int = 1000


In [5]:
ultrachat_200k_data = MsDataset.load('HuggingFaceH4/ultrachat_200k')

In [6]:
len(ultrachat_200k_data['train_sft'])

207865

In [ ]:
def tokenize_and_format(data):
    input_ids = tokenizer.apply_chat_template(
        data,
        tokenize=True,
        add_generation_prompt=False,
        truncation=True,
        max_length=2500,
    )

    return input_ids

In [ ]:
data = ultrachat_200k_data['train_sft'][0]['messages']
data

In [ ]:
data.insert(
        0,
        {"content": "You are a helpful assistant", "role": "system"}
    )

In [ ]:
data

In [ ]:
input_ids = tokenize_and_format(data)
input_ids.keys()

In [ ]:
## 生成训练数据的tokenid
chosen_input_ids_list = []
i = 0
while True:
    data = ultrachat_200k_data['train_sft'][i]['messages']
    # 添加 **系统提示词**
    data.insert(
        0,
        {"content": "You are a helpful assistant", "role": "system"}
    )
    input_ids = tokenize_and_format(data)
    if i == 0:
        print(tokenizer.decode(input_ids))
    chosen_input_ids_list.append(input_ids)
    i += 1
    if i % 1000 == 0:
        print(f"已处理{i}条数据")
    if i == 5000:
        break

print('-' * 70)

In [ ]:
print(len(chosen_input_ids_list))

In [ ]:
batch_size = SFTConfig.batch_size
gradient_accumulation_steps = SFTConfig.gradient_accumulation_steps
log_iter = SFTConfig.log_iter
max_lr = SFTConfig.max_lr
min_lr = SFTConfig.min_lr
warmup_steps = SFTConfig.warmup_steps
total_steps = len(chosen_input_ids_list) // batch_size
optimizer = torch.optim.AdamW(filter(
    lambda p: p.requires_grad,
    model.parameters()
), lr=max_lr)

In [ ]:
##配置logging
import time

with open(f"./Qwen3-0.6B-SFT_log.txt", "a") as my_file:
    my_file.write(f' \
        time:{time.strftime("%Y-%m-%d, %H:%M:%S")}, \
        batch_size:{batch_size}, \
        warmup_steps:{warmup_steps}, \
        max_lr:{max_lr}, \
        min_lr:{min_lr}\n')


# 定义一个日志记录函数
def log_call(iters, iters_average_loss):
    with open(f"./Qwen2.5-0.5B-SFT_log.txt", "a") as my_file:
        my_file.write(f' \
            time:{time.strftime("%Y-%m-%d, %H:%M:%S")}, \
            iters:{iters + 1}, \
            iters_average_Loss:{iters_average_loss:.4f}\n')


def linear_warmup(current_step, warmup_steps, max_lr):
    if current_step < warmup_steps:
        return max_lr * current_step / warmup_steps
    else:
        return max_lr


def cosine_decay(current_step, warmup_steps, total_steps, max_lr, min_lr):
    if current_step < warmup_steps:
        return linear_warmup(current_step, warmup_steps, max_lr)
    else:
        progress = (current_step - warmup_steps) \
                   / \
                   (total_steps - warmup_steps)
        decay = 0.5 * (1 + np.cos(np.pi * progress))
        return (max_lr - min_lr) * decay + min_lr